In [ ]:
import tensorflow as tf

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()  # TPU detection
    print('Running on TPU ', tpu.master())
except ValueError:
    raise BaseException('ERROR: Not connected to a TPU runtime; please see the previous cell in this notebook for instructions!')

tf.config.experimental_connect_to_cluster(tpu)
tf.tpu.experimental.initialize_tpu_system(tpu)
strategy = tf.distribute.TPUStrategy(tpu)  # Use the non-experimental symbol

# Print TPU details
print("TPUClusterResolver:", tpu)
print("Number of replicas:", strategy.num_replicas_in_sync)

Running on TPU  
TPUClusterResolver: <tensorflow.python.distribute.cluster_resolver.tpu.tpu_cluster_resolver.TPUClusterResolver object at 0x7ba9ae5a7490>
Number of replicas: 8


In [ ]:
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, LSTM, Embedding, Dense
import numpy as np
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
import os

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import configuration file for paths
import sys
sys.path.insert(0, '/content/drive/MyDrive/CSCK507_EMA_GroupC_Final')
import config

# Set the paths to the saved training, validation, and test sets
train_data_file = os.path.join(config.MAIN_DIR_PATH, config.TRAIN_DATA)
val_data_file = os.path.join(config.MAIN_DIR_PATH, config.VAL_DATA)
test_data_file = os.path.join(config.MAIN_DIR_PATH, config.TEST_DATA)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Mixed precision setup for TPU
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_bfloat16')
mixed_precision.set_global_policy(policy)

In [ ]:
# Load the Train & Validation Datasets
train_data = np.load(train_data_file)
X_train_enc, X_train_dec, y_train = train_data['X_train_enc'], train_data['X_train_dec'], train_data['y_train']

val_data = np.load(val_data_file)
X_val_enc, X_val_dec, y_val = val_data['X_val_enc'], val_data['X_val_dec'], val_data['y_val']

In [ ]:
# Initialize the TPU Strategy
# ---------------------------------------------------------------
resolver = tf.distribute.cluster_resolver.TPUClusterResolver()  # Automatically detects the TPU
print(f"TPU Address: {resolver.master()}")  # Print the TPU address for confirmation
strategy = tf.distribute.TPUStrategy(resolver)  # Create TPU strategy for parallel training

# Print detected TPU devices to confirm initialization
print("Logical Devices:", tf.config.list_logical_devices())

TPU Address: 
Logical Devices: [LogicalDevice(name='/device:CPU:0', device_type='CPU'), LogicalDevice(name='/device:TPU:0', device_type='TPU'), LogicalDevice(name='/device:TPU:1', device_type='TPU'), LogicalDevice(name='/device:TPU:2', device_type='TPU'), LogicalDevice(name='/device:TPU:3', device_type='TPU'), LogicalDevice(name='/device:TPU:4', device_type='TPU'), LogicalDevice(name='/device:TPU:5', device_type='TPU'), LogicalDevice(name='/device:TPU:6', device_type='TPU'), LogicalDevice(name='/device:TPU:7', device_type='TPU'), LogicalDevice(name='/device:TPU_SYSTEM:0', device_type='TPU_SYSTEM')]


In [ ]:
# Define the Distribution Strategy and Model
with strategy.scope():
    # Define Model Parameters
    max_length = X_train_enc.shape[1]
    vocab_size = int(np.max([np.max(X_train_enc), np.max(X_train_dec), np.max(y_train)])) + 1

    # Define Encoder Model with Smaller Embedding Dimensions
    encoder_inputs = Input(shape=(max_length,), name='Encoder-Input')
    encoder_embedding = Embedding(input_dim=vocab_size, output_dim=64, mask_zero=True, name='Encoder-Embedding')(encoder_inputs)
    encoder_lstm, state_h, state_c = LSTM(64, return_state=True, name='Encoder-LSTM')(encoder_embedding)

    # Define Decoder Model with Gradient Clipping
    decoder_inputs = Input(shape=(max_length,), name='Decoder-Input')
    decoder_embedding = Embedding(input_dim=vocab_size, output_dim=64, mask_zero=True, name='Decoder-Embedding')(decoder_inputs)
    decoder_lstm, _, _ = LSTM(64, return_sequences=True, return_state=True, name='Decoder-LSTM')(decoder_embedding, initial_state=[state_h, state_c])
    decoder_dense = Dense(vocab_size, activation='softmax', name='Decoder-Output')(decoder_lstm)

    # Compile the Model with Gradient Clipping
    optimizer = tf.keras.optimizers.Adam(clipvalue=1.0)  # Use gradient clipping to prevent large gradients
    seq2seq_model = Model([encoder_inputs, decoder_inputs], decoder_dense)
    seq2seq_model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
max_length

30

In [ ]:
vocab_size

640526

In [ ]:
# Define Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

In [ ]:
# Train the model using smaller batch size for stability of epochs
history = seq2seq_model.fit(
    [X_train_enc, X_train_dec], y_train,
    epochs=3,
    batch_size=64,
    validation_data=([X_val_enc, X_val_dec], y_val),
    callbacks=[early_stopping, lr_scheduler],
    verbose=1
)

Epoch 1/3
147685/147685 [==============================] - 12289s 82ms/step - loss: 4.7921 - accuracy: 0.2223 - val_loss: 4.6091 - val_accuracy: 0.2346 - lr: 0.0010
Epoch 2/3
147685/147685 [==============================] - 11919s 81ms/step - loss: 4.5704 - accuracy: 0.2372 - val_loss: 4.5683 - val_accuracy: 0.2381 - lr: 0.0010
Epoch 3/3
147685/147685 [==============================] - 11920s 81ms/step - loss: 4.5271 - accuracy: 0.2402 - val_loss: 4.5549 - val_accuracy: 0.2390 - lr: 0.0010


In [ ]:
# Save the Final Model
final_model_path = os.path.join(config.MAIN_DIR_PATH, config.SEQ2SEQ_NA_MODEL)
seq2seq_model.save(final_model_path)
print(f"Model saved successfully at {final_model_path}")

# Validate the Final Model
val_loss, val_acc = seq2seq_model.evaluate([X_val_enc, X_val_dec], y_val, verbose=1)
print(f"Final Validation Loss: {val_loss}, Final Validation Accuracy: {val_acc}")

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model saved successfully at /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/seq2seq_no_attention_model.h5
63294/63294 [==============================] - 1453s 22ms/step - loss: 4.5462 - accuracy: 0.2390
Final Validation Loss: 4.546170234680176, Final Validation Accuracy: 0.23904351890087128
